In [70]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config('spark.shuffle.useOldFetchProtocol','true'). \
config("spark.sql.warehouse.dir", f"/user/itv026931/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [71]:
from pyspark.sql.functions import *

In [72]:
spark.conf.set("spark.sql.unacceptable_rated_pts",0)
spark.conf.set("spark.sql.very_bad_rated_pts",100)
spark.conf.set("spark.sql.bad_rated_pts",200)
spark.conf.set("spark.sql.good_rated_pts",500)
spark.conf.set("spark.sql.very_good_rated_pts",650)
spark.conf.set("spark.sql.excellent_rated_pts",800)

In [73]:
spark.conf.set("spark.sql.unacceptable_grade_pts", 750)
spark.conf.set("spark.sql.very_bad_grade_pts", 1000)
spark.conf.set("spark.sql.bad_grade_pts", 1500)
spark.conf.set("spark.sql.good_grade_pts", 2000)
spark.conf.set("spark.sql.very_good_grade_pts", 2500)

LOAN SCORE CALCULATION CRITERIA 1-Payment History [Ph]

In [74]:
bad_customers_data_final_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema",True) \
.load("/user/itv026931/lendingclubproject/bad/bad_customers_data_final_df")

In [75]:
bad_customers_data_final_df.createOrReplaceTempView("bad_data_customer")

In [6]:
spark.sql("select * from itv026931_lending_club.loans_repayments")

loan_id,total_principal_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,ingest_date
76003861,6423.8,5607.85,0.0,12031.65,574.88,Jan-2018,null,2026-09-29 01:24:...
76263914,2400.0,323.18,0.0,2723.1802,905.18,Dec-2017,null,2026-09-29 01:24:...
75537401,12600.0,1392.76,0.0,13992.757,3437.74,Aug-2018,null,2026-09-29 01:24:...
75038986,7805.03,7583.71,0.0,15388.74,440.72,Mar-2019,Apr-2019,2026-09-29 01:24:...
76301424,2614.78,1080.09,0.0,3694.87,153.89,Apr-2018,null,2026-09-29 01:24:...
75333198,8664.09,3431.32,68.8,12164.21,343.9,Mar-2019,Apr-2019,2026-09-29 01:24:...
76391453,35000.0,10266.01,0.0,45266.008,18884.35,Feb-2019,null,2026-09-29 01:24:...
76363364,15000.0,2093.11,0.0,17093.107,4648.59,Aug-2018,null,2026-09-29 01:24:...
76272510,14463.21,11178.79,110.13,25752.13,734.18,Mar-2019,Apr-2019,2026-09-29 01:24:...
76304116,4800.0,1051.65,0.0,5851.6523,1761.59,Dec-2017,null,2026-09-29 01:24:...


In [7]:
spark.sql("select * from itv026931_lending_club.loans")

loan_id,member_id,loan_amount,funded_amount,loan_term_years,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,loan_title,ingest_date
6300587,e9f690be7e25fc285...,12000.0,12000.0,3,15.22,417.28,Jul-2013,Fully Paid,credit_card,Credit Card Payoff,2026-09-27 22:58:...
6311418,7602c853409dfc8d8...,7125.0,7125.0,3,15.88,250.08,Jul-2013,Fully Paid,other,Other,2026-09-27 22:58:...
6321535,5e1091070269a95ad...,24000.0,24000.0,5,14.33,562.56,Aug-2013,Fully Paid,credit_card,Loan for life,2026-09-27 22:58:...
6300526,cace2e77b60e12156...,24000.0,24000.0,3,6.03,730.46,Jul-2013,Fully Paid,credit_card,Refinance loan,2026-09-27 22:58:...
6301469,54f7c754fbe1cdad5...,18000.0,18000.0,5,17.56,452.79,Aug-2013,Fully Paid,debt_consolidation,Debt consolidation,2026-09-27 22:58:...
6306881,1e13cab7268220e9e...,20000.0,20000.0,5,11.55,440.36,Aug-2013,Fully Paid,debt_consolidation,Debt Consolidation,2026-09-27 22:58:...
6301579,6c8dfadce31c466b6...,12000.0,12000.0,3,11.55,396.0,Jul-2013,Fully Paid,debt_consolidation,Goodbye Stress,2026-09-27 22:58:...
6291661,2b0dcdeaf82ddc95c...,28000.0,28000.0,5,16.78,692.57,Aug-2013,Fully Paid,debt_consolidation,Debt consolidation,2026-09-27 22:58:...
6311612,882d6a9a903c44fc2...,5500.0,5500.0,3,10.64,179.13,Aug-2013,Charged Off,credit_card,Yes I Can Manage ...,2026-09-27 22:58:...
6311704,946424debb78b996f...,23600.0,23600.0,5,12.35,529.16,Jul-2013,Fully Paid,debt_consolidation,Consolidation,2026-09-27 22:58:...


In [76]:
ph_df = spark.sql("select c.member_id, \
case \
when p.last_payment_amount < (c.monthly_installment * 0.5) then ${spark.sql.very_bad_rated_pts} \
when p.last_payment_amount >= (c.monthly_installment * 0.5) and p.last_payment_amount < c.monthly_installment then ${spark.sql.very_bad_rated_pts} \
when (p.last_payment_amount = (c.monthly_installment)) then ${spark.sql.good_rated_pts} \
when p.last_payment_amount > (c.monthly_installment) and p.last_payment_amount <= (c.monthly_installment * 1.50) then ${spark.sql.very_good_rated_pts} \
when p.last_payment_amount  > (c.monthly_installment * 1.50) then ${spark.sql.excellent_rated_pts} \
else ${spark.sql.unacceptable_rated_pts} \
end as last_payment_pts, \
case \
when p.total_payment_received >= (c.funded_amount * 0.50) then ${spark.sql.very_good_grade_pts} \
when p.total_payment_received < (c.funded_amount * 0.50) and p.total_payment_received > 0 then ${spark.sql.good_rated_pts} \
when p.total_payment_received = 0 or (p.total_payment_received) is null then ${spark.sql.unacceptable_rated_pts} \
end as total_payment_pts \
from itv026931_lending_club.loans_repayments as p \
inner join \
itv026931_lending_club.loans as c \
on p.loan_id = c.loan_id where member_id NOT IN (select member_id from bad_data_customer)")

In [9]:
ph_df

member_id,last_payment_pts,total_payment_pts
efc702f2aede7abcd...,800,2500
0379e28648ddb750a...,500,2500
368e070d2646b3ea7...,500,2500
26dc2ba8814069c77...,800,2500
0d5707d3064ca5476...,500,2500
9166cd82d7caf416c...,800,2500
0278ec25185196740...,800,2500
3ed5058362dbf8560...,800,2500
4c3c0cc7f8e1a635e...,500,2500
49e1a02139b7f5e6d...,500,2500


In [77]:
ph_df.createOrReplaceTempView("ph_pts")

LOAN SCORE CALCULATION CRITERIA 2: LOAN DEFAULTERS HISTORY(ldh)

In [78]:
ldh_ph_df = spark.sql("""
select p.*,
CASE
WHEN d.delinq_2yrs = 0 THEN ${spark.sql.excellent_rated_pts} \
WHEN d.delinq_2yrs BETWEEN 1 and 2 THEN ${spark.sql.bad_rated_pts} \
WHEN d.delinq_2yrs BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
WHEN d.delinq_2yrs > 5 OR d.delinq_2yrs IS NULL THEN ${spark.sql.unacceptable_grade_pts} \
END AS delinq_pts, \
CASE \
WHEN l.pub_rec = 0 THEN ${spark.sql.excellent_rated_pts} \
WHEN l.pub_rec BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
WHEN l.pub_rec BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
WHEN l.pub_rec > 5 OR l.pub_rec IS NULL THEN ${spark.sql.very_bad_rated_pts} \
END AS public_records_pts, \
CASE \
WHEN l.pub_rec_bankruptcies = 0 THEN ${spark.sql.excellent_rated_pts} \
WHEN l.pub_rec_bankruptcies BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
WHEN l.pub_rec_bankruptcies BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
WHEN l.pub_rec_bankruptcies > 5 OR l.pub_rec_bankruptcies IS NULL THEN ${spark.sql.very_bad_rated_pts} \
END as public_bankruptcies_pts, \
CASE \
WHEN l.inq_last_6mths = 0 THEN ${spark.sql.excellent_rated_pts} \
WHEN l.inq_last_6mths BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
WHEN l.inq_last_6mths BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
WHEN l.inq_last_6mths > 5 OR l.inq_last_6mths IS NULL THEN ${spark.sql.unacceptable_rated_pts} \
END AS enq_pts \
FROM itv026931_lending_club.loans_defaulters_detail_rec_enq_nar_new l \
INNER JOIN itv026931_lending_club.loans_defaulters_delinq_new d ON d.member_id = l.member_id  \
INNER JOIN ph_pts p ON p.member_id = l.member_id where l.member_id NOT IN (select member_id from bad_data_customer)
""")

In [79]:
ldh_ph_df.createOrReplaceTempView("ldh_ph_pts")

In [80]:
fh_ldh_ph_df = spark.sql("select ldef.*, \
   CASE \
   WHEN LOWER(l.loan_status) LIKE '%fully paid%' THEN ${spark.sql.excellent_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%current%' THEN ${spark.sql.good_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%in grace period%' THEN ${spark.sql.bad_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%late (16-30 days)%' OR LOWER(l.loan_status) LIKE '%late (31-120 days)%' THEN ${spark.sql.very_bad_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%charged off%' THEN ${spark.sql.unacceptable_rated_pts} \
   else ${spark.sql.unacceptable_rated_pts} \
   END AS loan_status_pts, \
   CASE \
   WHEN LOWER(a.home_ownership) LIKE '%own' THEN ${spark.sql.excellent_rated_pts} \
   WHEN LOWER(a.home_ownership) LIKE '%rent' THEN ${spark.sql.good_rated_pts} \
   WHEN LOWER(a.home_ownership) LIKE '%mortgage' THEN ${spark.sql.bad_rated_pts} \
   WHEN LOWER(a.home_ownership) LIKE '%any' OR LOWER(a.home_ownership) IS NULL THEN ${spark.sql.very_bad_rated_pts} \
   END AS home_pts, \
   CASE \
   WHEN l.funded_amount <= (a.total_high_credit_limit * 0.10) THEN ${spark.sql.excellent_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.10) AND l.funded_amount <= (a.total_high_credit_limit * 0.20) THEN ${spark.sql.very_good_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.20) AND l.funded_amount <= (a.total_high_credit_limit * 0.30) THEN ${spark.sql.good_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.30) AND l.funded_amount <= (a.total_high_credit_limit * 0.50) THEN ${spark.sql.bad_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.50) AND l.funded_amount <= (a.total_high_credit_limit * 0.70) THEN ${spark.sql.very_bad_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.70) THEN ${spark.sql.unacceptable_rated_pts} \
   else ${spark.sql.unacceptable_rated_pts} \
   END AS credit_limit_pts, \
   CASE \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A1' THEN ${spark.sql.excellent_rated_pts} \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A2' THEN (${spark.sql.excellent_rated_pts} * 0.95) \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A3' THEN (${spark.sql.excellent_rated_pts} * 0.90) \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A4' THEN (${spark.sql.excellent_rated_pts} * 0.85) \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A5' THEN (${spark.sql.excellent_rated_pts} * 0.80) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B1' THEN (${spark.sql.very_good_rated_pts}) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B2' THEN (${spark.sql.very_good_rated_pts} * 0.95) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B3' THEN (${spark.sql.very_good_rated_pts} * 0.90) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B4' THEN (${spark.sql.very_good_rated_pts} * 0.85) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B5' THEN (${spark.sql.very_good_rated_pts} * 0.80) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C1' THEN (${spark.sql.good_rated_pts}) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C2' THEN (${spark.sql.good_rated_pts} * 0.95) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C3' THEN (${spark.sql.good_rated_pts} * 0.90) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C4' THEN (${spark.sql.good_rated_pts} * 0.85) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C5' THEN (${spark.sql.good_rated_pts} * 0.80) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D1' THEN (${spark.sql.bad_rated_pts}) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D2' THEN (${spark.sql.bad_rated_pts} * 0.95) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D3' THEN (${spark.sql.bad_rated_pts} * 0.90) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D4' THEN (${spark.sql.bad_rated_pts} * 0.85) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D5' THEN (${spark.sql.bad_rated_pts} * 0.80) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E1' THEN (${spark.sql.very_bad_rated_pts}) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E2' THEN (${spark.sql.very_bad_rated_pts} * 0.95) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E3' THEN (${spark.sql.very_bad_rated_pts} * 0.90) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E4' THEN (${spark.sql.very_bad_rated_pts} * 0.85) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E5' THEN (${spark.sql.very_bad_rated_pts} * 0.80) \
   WHEN (a.grade) in ('F', 'G') THEN (${spark.sql.unacceptable_rated_pts}) \
   END AS grade_pts \
   FROM ldh_ph_pts ldef \
   INNER JOIN itv026931_lending_club.loans l ON ldef.member_id = l.member_id \
   INNER JOIN itv026931_lending_club.customers_new a ON a.member_id = ldef.member_id where ldef.member_id NOT IN (select member_id from bad_data_customer)")

In [81]:
fh_ldh_ph_df.createOrReplaceTempView("fh_ldh_ph_pts")

In [82]:
fh_ldh_ph_df

member_id,last_payment_pts,total_payment_pts,delinq_pts,public_records_pts,public_bankruptcies_pts,enq_pts,loan_status_pts,home_pts,credit_limit_pts,grade_pts
002cb5eb3fff0d30d...,500,500,800,800,800,800,500,200,800,85.00
0036ecd43123158e6...,500,500,200,200,200,100,500,200,800,552.50
003eb3c570303d952...,650,2500,750,800,800,800,800,500,800,520.00
0054f9c64056de774...,500,500,200,800,800,800,0,500,0,170.00
00553d7d341fe64b5...,800,2500,200,800,800,200,800,200,650,400.00
007f68f5d0df233a4...,100,2500,800,800,800,200,800,500,100,100.00
008ec47749e2f71c7...,800,2500,800,800,800,200,800,200,800,200.00
0097a16f2eef3cd5c...,800,2500,800,800,800,800,800,500,200,100.00
0098979a879ee61fb...,500,500,800,200,200,800,500,200,200,650.00
009bd3b00bd4fe618...,800,2500,800,800,800,800,800,500,200,640.00


In [19]:
spark.sql("select * from fh_ldh_ph_pts")

member_id,last_payment_pts,total_payment_pts,delinq_pts,public_records_pts,public_bankruptcies_pts,enq_pts,loan_status_pts,home_pts,credit_limit_pts,grade_pts
002cb5eb3fff0d30d...,500,500,800,800,800,800,500,200,800,85.00
0036ecd43123158e6...,500,500,200,200,200,100,500,200,800,552.50
003eb3c570303d952...,650,2500,750,800,800,800,800,500,800,520.00
0054f9c64056de774...,500,500,200,800,800,800,0,500,0,170.00
00553d7d341fe64b5...,800,2500,200,800,800,200,800,200,650,400.00
007f68f5d0df233a4...,100,2500,800,800,800,200,800,500,100,100.00
008ec47749e2f71c7...,800,2500,800,800,800,200,800,200,800,200.00
0097a16f2eef3cd5c...,800,2500,800,800,800,800,800,500,200,100.00
0098979a879ee61fb...,500,500,800,200,200,800,500,200,200,650.00
009bd3b00bd4fe618...,800,2500,800,800,800,800,800,500,200,640.00


Final loan score calculation by considering all the 3 criterias with the following %**
1. Payment_History = 20% [Considering the columns last_payment_pts, total_payment_pts]
2. Loan Defaults = 45% [Considering the columns delinq_pts, public_records_pts, public_bankruptcies_pts, enq_pts]
3. Financial Health = 35% [Considering the columns loan_status_pts, home_pts, credit_limit_pts, grade_pts]

In [83]:
loan_score = fh_ldh_ph_df.withColumn("payment_history_pts",(col("last_payment_pts")+col("total_payment_pts")*0.20)) \
.withColumn("defaulters_history_pts",(col("delinq_pts")+col("public_records_pts")+col("public_bankruptcies_pts")+col("enq_pts")*0.45)) \
.withColumn("financial_health_pts",(col("loan_status_pts")+col("home_pts")+col("credit_limit_pts")+col("grade_pts")*0.35)) \
.select("member_id","payment_history_pts","defaulters_history_pts","financial_health_pts")

In [84]:
loan_score

member_id,payment_history_pts,defaulters_history_pts,financial_health_pts
002cb5eb3fff0d30d...,600.0,2760.0,1529.75
0036ecd43123158e6...,600.0,645.0,1693.375
003eb3c570303d952...,1150.0,2710.0,2282.0
0054f9c64056de774...,600.0,2160.0,559.5
00553d7d341fe64b5...,1300.0,1890.0,1790.0
007f68f5d0df233a4...,600.0,2490.0,1435.0
008ec47749e2f71c7...,1300.0,2490.0,1870.0
0097a16f2eef3cd5c...,1300.0,2760.0,1535.0
0098979a879ee61fb...,600.0,1560.0,1127.5
009bd3b00bd4fe618...,1300.0,2760.0,1724.0


In [85]:
final_loan_score = loan_score.withColumn("loan_score",(col("payment_history_pts")+col("defaulters_history_pts")+col("financial_health_pts")))

In [86]:
final_loan_score.createOrReplaceTempView("loan_score_eval")

In [87]:
loan_score_final = spark.sql("""
select ls.*,
case
when loan_score > ${spark.sql.very_good_grade_pts} then 'A'
when loan_score <= ${spark.sql.very_good_grade_pts} AND loan_score > ${spark.sql.good_grade_pts} then 'B'
when loan_score <= ${spark.sql.good_grade_pts} AND loan_score > ${spark.sql.bad_grade_pts} THEN 'C'
when loan_score <= ${spark.sql.bad_grade_pts} AND loan_score  > ${spark.sql.very_bad_grade_pts} THEN 'D'
when loan_score <= ${spark.sql.very_bad_grade_pts} AND loan_score > ${spark.sql.unacceptable_grade_pts} THEN 'E'
when loan_score <= ${spark.sql.unacceptable_grade_pts} THEN 'F'
end as loan_final_grade
from loan_score_eval ls
""")

In [88]:
loan_score_final.createOrReplaceTempView("loan_final_table")

In [89]:
loan_score_final.count()

1102587

In [92]:
loan_score_final.select("loan_final_grade").distinct()

loan_final_grade
null
B
D
C
A


In [96]:
loan_score_final.filter("loan_score >= 2000 and loan_score <= 2500")

member_id,payment_history_pts,defaulters_history_pts,financial_health_pts,loan_score,loan_final_grade
1ecd9723143ec85be...,200.0,1090.0,916.125,2206.125,B
2043b261277540f47...,600.0,590.0,1166.25,2356.25,B
21c2079873d3b477f...,600.0,960.0,859.5,2419.5,B
255bc530ebced01eb...,200.0,960.0,866.25,2026.25,B
2e390fb4ba1b39add...,600.0,960.0,463.0,2023.0,B
462cb634b818c49b1...,600.0,1290.0,431.5,2321.5,B
6c9b47c3c8e398e35...,600.0,960.0,729.75,2289.75,B
8180d7b88fd8919aa...,600.0,690.0,1166.25,2456.25,B
9739683ed91c8fd3e...,1000.0,690.0,770.0,2460.0,B
b0fcb3d39c580b56e...,200.0,1290.0,990.0,2480.0,B


In [97]:
loan_score_final.write \
.format("parquet") \
.option("path","/user/itv026931/lendingclubproject/processed") \
.mode("overwrite") \
.save()
